# Module 3: Production Deployment with AgentCore

## Overview

In this module, you'll deploy the multi-agent customer service system to production using:

- **AgentCore Runtime**: Managed hosting for the agent application
- **AgentCore Gateway**: MCP tool integration (optional)
- **AgentCore Memory**: Conversation persistence
- **AgentCore Observability**: CloudWatch integration for tracing

### Learning Objectives
1. Deploy agents to AgentCore Runtime
2. Configure observability with CloudWatch
3. Set up session management with AgentCore Memory
4. Test the production deployment

### Time: ~60 minutes

## Step 1: Environment Setup

In [ ]:
# Install dependencies
!pip install -q bedrock-agentcore bedrock-agentcore-starter-toolkit boto3

In [ ]:
import boto3
import json
import os
import time
from datetime import datetime

# Get AWS configuration
session = boto3.Session()
REGION = session.region_name or 'us-east-1'
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

print(f"AWS Region: {REGION}")
print(f"Account ID: {ACCOUNT_ID}")

In [ ]:
# Load baseline metrics from Module 2
try:
    %store -r baseline_metrics
    print("Loaded baseline metrics from Module 2")
    print(f"Goal Success Rate: {baseline_metrics.get('goal_success_rate', 'N/A'):.2%}")
except:
    print("Could not load baseline metrics - will establish new baseline in production")
    baseline_metrics = {}

## Step 2: Create Requirements File

In [ ]:
%%writefile requirements.txt
bedrock-agentcore
strands-agents
boto3
aws-opentelemetry-distro

## Step 3: Review the Production Application

The `app.py` file contains our production-ready multi-agent system with:
- Inline tool definitions for single-file deployment
- Lazy agent initialization for cold start optimization
- Proper logging for observability
- AgentCore entrypoint decorator
- Global cross-region inference with Claude Sonnet 4.5 and Haiku 4.5

In [ ]:
# Review the app structure
with open('app.py', 'r') as f:
    content = f.read()

# Show key sections
print("Production Application Structure:")
print("="*50)
print("\n1. Tool Definitions (Order, Product, Account)")
print("2. Agent Creation Functions (Haiku 4.5 for sub-agents, Sonnet 4.5 for orchestrator)")
print("3. AgentCore Entrypoint with session handling")
print(f"\nTotal lines: {len(content.splitlines())}")

## Step 4: Create Execution Role

The execution role needs permissions for:
- Amazon Bedrock model invocation
- DynamoDB access for orders and accounts
- Bedrock Knowledge Base retrieval
- CloudWatch logging and X-Ray tracing

In [ ]:
import json

iam_client = boto3.client('iam')
AGENT_NAME = 'ecommerce-customer-service'
ROLE_NAME = f'AgentCore-{AGENT_NAME}-role'

# Trust policy for AgentCore
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

# Permissions policy
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockAccess",
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream"
            ],
            "Resource": "*"
        },
        {
            "Sid": "BedrockKBAccess",
            "Effect": "Allow",
            "Action": [
                "bedrock:Retrieve",
                "bedrock:RetrieveAndGenerate"
            ],
            "Resource": "*"
        },
        {
            "Sid": "DynamoDBAccess",
            "Effect": "Allow",
            "Action": [
                "dynamodb:GetItem",
                "dynamodb:PutItem",
                "dynamodb:UpdateItem",
                "dynamodb:Query",
                "dynamodb:Scan"
            ],
            "Resource": [
                f"arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/ecommerce-workshop-*"
            ]
        },
        {
            "Sid": "SSMAccess",
            "Effect": "Allow",
            "Action": ["ssm:GetParameter"],
            "Resource": f"arn:aws:ssm:{REGION}:{ACCOUNT_ID}:parameter/ecommerce-workshop-*"
        },
        {
            "Sid": "CloudWatchLogs",
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": "*"
        },
        {
            "Sid": "XRayTracing",
            "Effect": "Allow",
            "Action": [
                "xray:PutTraceSegments",
                "xray:PutTelemetryRecords"
            ],
            "Resource": "*"
        }
    ]
}

# Create role
try:
    response = iam_client.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Execution role for E-Commerce Customer Service Agent'
    )
    ROLE_ARN = response['Role']['Arn']
    print(f"Created role: {ROLE_ARN}")
    
    # Attach permissions
    iam_client.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName=f'{ROLE_NAME}-policy',
        PolicyDocument=json.dumps(permissions_policy)
    )
    print("Attached permissions policy")
    
    # Wait for role propagation
    print("Waiting for IAM role propagation...")
    time.sleep(10)
    
except iam_client.exceptions.EntityAlreadyExistsException:
    ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
    print(f"Role already exists: {ROLE_ARN}")

## Step 5: Deploy to AgentCore Runtime

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Initialize Runtime
runtime = Runtime()

# Configure deployment
config = runtime.configure(
    entrypoint="app.py",
    execution_role=ROLE_ARN,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=AGENT_NAME
)

print("Configuration complete:")
print(f"  Agent Name: {AGENT_NAME}")
print(f"  Region: {REGION}")
print(f"  Execution Role: {ROLE_ARN}")

In [ ]:
# Launch deployment
print("Starting deployment to AgentCore Runtime...")
print("This will:")
print("  1. Build container image")
print("  2. Push to ECR")
print("  3. Create AgentCore Runtime")
print("  4. Configure observability")
print("\nThis may take 5-10 minutes...\n")

launch_result = runtime.launch()
print(f"\nLaunch initiated!")

In [ ]:
# Wait for deployment to complete
print("Waiting for deployment to complete...")

terminal_states = {'READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED'}

while True:
    status_result = runtime.status()
    status = status_result.endpoint.get('status', 'UNKNOWN')
    
    print(f"  Status: {status}")
    
    if status in terminal_states:
        break
    
    time.sleep(15)

if status == 'READY':
    print("\n" + "="*50)
    print("DEPLOYMENT SUCCESSFUL!")
    print("="*50)
    print(f"Agent Runtime ID: {launch_result.agent_id}")
    print(f"Agent ARN: {launch_result.agent_arn}")
else:
    print(f"\nDeployment ended with status: {status}")

## Step 6: Test Production Deployment

In [ ]:
# Test the deployed agent
print("Testing Production Deployment")
print("="*50)

test_prompts = [
    "What's the status of order ORD-2024-10002?",
    "Do you have any wireless headphones under $100?",
    "What are the benefits of Gold membership?"
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\nTest {i}: {prompt}")
    print("-"*40)
    
    try:
        response = runtime.invoke({"prompt": prompt})
        print(f"Response: {response.get('response', response)[:500]}...")
    except Exception as e:
        print(f"Error: {e}")

## Step 7: Configure Observability

AgentCore automatically integrates with CloudWatch for observability. Let's verify the setup.

In [ ]:
# Check CloudWatch log group
logs_client = boto3.client('logs', region_name=REGION)

log_group_prefix = f'/aws/bedrock-agentcore/{AGENT_NAME}'

try:
    response = logs_client.describe_log_groups(logGroupNamePrefix=log_group_prefix)
    log_groups = response.get('logGroups', [])
    
    if log_groups:
        print("CloudWatch Log Groups:")
        for lg in log_groups:
            print(f"  - {lg['logGroupName']}")
    else:
        print(f"No log groups found with prefix: {log_group_prefix}")
        print("Log groups will be created on first invocation.")
except Exception as e:
    print(f"Error checking logs: {e}")

In [ ]:
# View recent traces (if available)
print("\nObservability Dashboard Links:")
print("="*50)
print(f"\nCloudWatch Console:")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core/agents")
print(f"\nX-Ray Traces:")
print(f"  https://{REGION}.console.aws.amazon.com/xray/home?region={REGION}#/traces")

## Step 8: Session Management with AgentCore Memory (Optional)

For multi-turn conversations, AgentCore provides built-in session management.

In [ ]:
# Test multi-turn conversation with session
import uuid

session_id = str(uuid.uuid4())
print(f"Starting conversation session: {session_id}")
print("="*50)

conversation = [
    "Hi, I'd like to check on my order",
    "The order number is ORD-2024-10002",
    "When will it arrive?"
]

for i, message in enumerate(conversation, 1):
    print(f"\nTurn {i}")
    print(f"Customer: {message}")
    
    try:
        response = runtime.invoke(
            {"prompt": message},
            session_id=session_id
        )
        print(f"Agent: {response.get('response', response)[:300]}...")
    except Exception as e:
        print(f"Error: {e}")

## Step 9: Save Deployment Information

In [ ]:
# Save deployment info for Module 4
deployment_info = {
    'agent_name': AGENT_NAME,
    'agent_id': launch_result.agent_id if 'launch_result' in dir() else None,
    'agent_arn': launch_result.agent_arn if 'launch_result' in dir() else None,
    'region': REGION,
    'deployment_time': datetime.now().isoformat(),
    'execution_role': ROLE_ARN
}

%store deployment_info
%store runtime
%store baseline_metrics
print("Deployment information saved for Module 4")

## Summary

In this module, you:

1. **Created execution role** with appropriate permissions for DynamoDB, Bedrock, and CloudWatch
2. **Deployed to AgentCore Runtime** using the Starter Toolkit
3. **Tested the production deployment** with sample queries
4. **Verified observability** with CloudWatch integration
5. **Tested session management** for multi-turn conversations

### Production Architecture

```
+---------------------------------------------------------+
|                    AgentCore Runtime                     |
|  +---------------------------------------------------+  |
|  |              Multi-Agent Application               |  |
|  |  +-----------+ +-----------+ +-----------+        |  |
|  |  |  Order    | |  Product  | |  Account  |        |  |
|  |  |  Agent    | |  Agent    | |  Agent    |        |  |
|  |  | (Haiku45) | | (Haiku45) | | (Haiku45) |        |  |
|  |  +-----+-----+ +-----+-----+ +-----+-----+        |  |
|  |        +-------------+-----------+                 |  |
|  |              +-------v-------+                     |  |
|  |              | Orchestrator  |                     |  |
|  |              | (Sonnet 4.5)  |                     |  |
|  |              +---------------+                     |  |
|  +---------------------------------------------------+  |
|                          |                              |
|  +-----------------------------------------------+     |
|  |              AgentCore Observability           |     |
|  |  CloudWatch Logs | X-Ray Traces | Metrics      |     |
|  +-----------------------------------------------+     |
+---------------------------------------------------------+
```

### Next Steps

In **Module 4**, we'll configure online evaluation, simulate drift scenarios, and implement the improvement loop.